# Step 2: Data Cleaning & Star Schema Transformation
**Chicago Food Safety Inspection Analytics System (FIAS)**

In this notebook, transform the raw data saved in Microsoft SQL Server into an analytical **Star Schema**:
* **Dimension Tables**: `dim_date`, `dim_facility_type`, `dim_geography`, `dim_establishment`
* **Fact Tables**: `fact_inspections`, `fact_violations`

All cleaned and normalized data will be stored directly into **Microsoft SQL Server (`FoodInspectionDB`)** for fast analytical reporting in SSMS and Power BI.


### 1. Import Libraries & Connect to Microsoft SQL Server
Use `pyodbc` and `pandas` to connect to local SQL Server instance (`localhost\SQLEXPRESS`).


In [1]:
import re
from datetime import datetime
import pandas as pd
import pyodbc

# SQL Server Connection Configuration
SERVER_NAME = r"localhost\SQLEXPRESS"
DATABASE_NAME = "FoodInspectionDB"

def get_odbc_driver():
    drivers = pyodbc.drivers()
    for drv in ["ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server", "SQL Server"]:
        if drv in drivers:
            return drv
    return "ODBC Driver 18 for SQL Server"

def get_connection():
    driver = get_odbc_driver()
    conn_str = f"DRIVER={{{driver}}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};Trusted_Connection=yes;"
    if "18" in driver:
        conn_str += "TrustServerCertificate=yes;"
    return pyodbc.connect(conn_str)

# Test connection
conn = get_connection()
print(f"[+] Connected successfully to SQL Server: {SERVER_NAME} | Database: {DATABASE_NAME}")
conn.close()


[+] Connected successfully to SQL Server: localhost\SQLEXPRESS | Database: FoodInspectionDB


### 2. Load Raw Staging Data from SQL Server
Let's read the raw records that were saved in Step 1 (`raw_inspections` and `raw_violations`).


In [2]:
conn = get_connection()

df_inspections = pd.read_sql("SELECT * FROM raw_inspections", conn)
df_violations = pd.read_sql("SELECT * FROM raw_violations", conn)

conn.close()

print(f"[*] Loaded {len(df_inspections)} raw inspection records.")
print(f"[*] Loaded {len(df_violations)} raw violation records.")
df_inspections.head(3)


[*] Loaded 5000 raw inspection records.
[*] Loaded 13990 raw violation records.


C:\Users\muiva\AppData\Local\Temp\ipykernel_24332\1945713964.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_inspections = pd.read_sql("SELECT * FROM raw_inspections", conn)
C:\Users\muiva\AppData\Local\Temp\ipykernel_24332\1945713964.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_violations = pd.read_sql("SELECT * FROM raw_violations", conn)


,inspection_id,dba_name,aka_name,license_number,facility_type,risk_level,address,city,state,zip_code,inspection_date,inspection_type,results,latitude,longitude
0,2637016,EL TACO CART,EL TACO CART,2924487,Restaurant,Risk 1 (High),3207 N HALSTED ST,CHICAGO,IL,60657,2026-05-18,Non-Inspection,No Entry,41.940201,-87.649126
1,2637017,WILLIAMS INN PIZZERIA,WILLIAMS INN PIZZERIA,2616298,Restaurant,Risk 1 (High),2210 S MICHIGAN AVE,CHICAGO,IL,60616,2026-05-18,Complaint,Pass,41.852504,-87.623942
2,2637018,EGG TUCK,EGG TUCK,2938350,Restaurant,Risk 1 (High),326 N MICHIGAN AVE,CHICAGO,IL,60601,2026-05-18,Canvass,Fail,41.887415,-87.624698


### 3. Cleaning & Standardization Functions

Implement the business logic required:
1. Standardize facility types into top categories (Restaurant, Grocery, School, Bakery, etc.).
2. Validate Chicago GPS bounding coordinates (41.60 < lat < 42.10, -87.95 < lon < -87.50).
3. Standardize Risk levels (High, Medium, Low, Unassigned).
4. Standardize Inspection Types into clean analytical categories.


In [3]:
def clean_text(text):
    if not text or pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text).strip()).upper()

def standardize_facility_type(raw_type):
    if not raw_type or pd.isna(raw_type):
        return "Other / General Food Service"
    val = clean_text(raw_type)
    if "RESTAURANT" in val:
        return "Restaurant"
    elif "GROCERY" in val or "SUPERMARKET" in val:
        return "Grocery Store"
    elif any(k in val for k in ["SCHOOL", "DAYCARE", "CHILD", "HEAD START"]):
        return "School / Daycare"
    elif "BAKERY" in val:
        return "Bakery"
    elif "MOBILE" in val or "FOOD TRUCK" in val or "CART" in val:
        return "Mobile Food Vendor"
    elif any(k in val for k in ["LIQUOR", "TAVERN", "BAR", "LOUNGE", "PUB", "BREW"]):
        return "Bar / Tavern"
    elif "WHOLESALE" in val or "DISTRIBUTOR" in val or "COMMISSARY" in val:
        return "Wholesale / Distribution"
    elif any(k in val for k in ["HOSPITAL", "NURSING", "HEALTH", "LONG TERM"]):
        return "Healthcare Facility"
    elif "SPECIAL EVENT" in val or "POPUP" in val or "POPUPS" in val:
        return "Pop-up / Special Event"
    else:
        return "Other / General Food Service"

def standardize_inspection_type(raw_type):
    if not raw_type or pd.isna(raw_type):
        return "Other Administrative"
    val = clean_text(raw_type)
    if "CANVASS" in val and "RE-INSPECTION" in val:
        return "Canvass Re-Inspection"
    elif "CANVASS" in val:
        return "Routine Canvass"
    elif "COMPLAINT" in val and "RE-INSPECTION" in val:
        return "Complaint Re-Inspection"
    elif "COMPLAINT" in val:
        return "Complaint"
    elif "LICENSE" in val:
        return "License Initial / Renewal"
    elif "FOOD POISONING" in val or "SUSPECT" in val:
        return "Suspected Food Poisoning"
    else:
        return "Other Administrative"

def standardize_risk(raw_risk):
    if not raw_risk or pd.isna(raw_risk):
        return "Unassigned"
    val = str(raw_risk).strip().upper()
    if "1" in val or "HIGH" in val:
        return "High"
    elif "2" in val or "MEDIUM" in val:
        return "Medium"
    elif "3" in val or "LOW" in val:
        return "Low"
    return "Unassigned"

def validate_coordinates(lat, lon):
    try:
        lat = float(lat)
        lon = float(lon)
        if 41.60 <= lat <= 42.10 and -87.95 <= lon <= -87.50:
            return round(lat, 6), round(lon, 6)
    except (ValueError, TypeError):
        pass
    return None, None



### 4. Create Star Schema Tables in SQL Server
Ensure that `dim_date`, `dim_facility_type`, `dim_geography`, `dim_establishment`, `fact_inspections`, and `fact_violations` are created in Microsoft SQL Server.


In [5]:
conn = get_connection()
cur = conn.cursor()

# 1. Dimension Tables
cur.execute("""
IF OBJECT_ID('dim_date', 'U') IS NULL
CREATE TABLE dim_date (
    date_key INT PRIMARY KEY,
    full_date DATE NOT NULL,
    year INT NOT NULL,
    quarter INT NOT NULL,
    month INT NOT NULL,
    month_name VARCHAR(20) NOT NULL,
    day_of_week INT NOT NULL,
    day_name VARCHAR(20) NOT NULL,
    is_weekend BIT NOT NULL
);

IF OBJECT_ID('dim_facility_type', 'U') IS NULL
CREATE TABLE dim_facility_type (
    facility_type_key INT IDENTITY(1,1) PRIMARY KEY,
    raw_name VARCHAR(150) NOT NULL,
    standardized_category VARCHAR(100) NOT NULL,
    CONSTRAINT uq_facility_type UNIQUE (raw_name)
);

IF OBJECT_ID('dim_geography', 'U') IS NULL
CREATE TABLE dim_geography (
    geography_key INT IDENTITY(1,1) PRIMARY KEY,
    street_address VARCHAR(255) NOT NULL,
    city VARCHAR(50) DEFAULT 'CHICAGO',
    state VARCHAR(10) DEFAULT 'IL',
    zip_code VARCHAR(10),
    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6),
    CONSTRAINT uq_geography UNIQUE (street_address, zip_code)
);

IF OBJECT_ID('dim_establishment', 'U') IS NULL
CREATE TABLE dim_establishment (
    establishment_key INT IDENTITY(1,1) PRIMARY KEY,
    license_number INT NOT NULL,
    dba_name VARCHAR(255) NOT NULL,
    aka_name VARCHAR(255),
    risk_level VARCHAR(50),
    first_inspection_date DATE,
    last_inspection_date DATE,
    CONSTRAINT uq_establishment UNIQUE (license_number, dba_name)
);
""")
conn.commit()

# 2. Fact Tables
cur.execute("""
IF OBJECT_ID('fact_inspections', 'U') IS NULL
CREATE TABLE fact_inspections (
    inspection_id BIGINT PRIMARY KEY,
    date_key INT FOREIGN KEY REFERENCES dim_date(date_key),
    establishment_key INT FOREIGN KEY REFERENCES dim_establishment(establishment_key),
    geography_key INT FOREIGN KEY REFERENCES dim_geography(geography_key),
    facility_type_key INT FOREIGN KEY REFERENCES dim_facility_type(facility_type_key),
    inspection_type VARCHAR(100) NOT NULL,
    results VARCHAR(50) NOT NULL,
    is_failure BIT NOT NULL,
    priority_violations_count INT DEFAULT 0,
    serious_violations_count INT DEFAULT 0,
    core_violations_count INT DEFAULT 0,
    total_violations_count INT DEFAULT 0,
    days_since_last_inspection INT NULL,
    prior_failures_cumulative INT DEFAULT 0
);

IF OBJECT_ID('fact_violations', 'U') IS NULL
CREATE TABLE fact_violations (
    violation_id BIGINT IDENTITY(1,1) PRIMARY KEY,
    inspection_id BIGINT FOREIGN KEY REFERENCES fact_inspections(inspection_id) ON DELETE CASCADE,
    violation_code INT NOT NULL,
    violation_description VARCHAR(255) NOT NULL,
    severity_tier VARCHAR(50) NOT NULL,
    severity_weight INT NOT NULL,
    inspector_comment NVARCHAR(MAX)
);
""")
conn.commit()
conn.close()

### 5. Clean & Populate the Dimension Tables
extract distinct values from `df_inspections` for each dimension and insert them into SQL Server.


In [6]:
conn = get_connection()
cur = conn.cursor()
cur.fast_executemany = True

# --- 5.1 Date Dimension ---
cur.execute("SELECT date_key FROM dim_date")
existing_date_keys = {r[0] for r in cur.fetchall()}

date_rows = []
for d_val in df_inspections['inspection_date'].dropna().unique():
    dt = pd.to_datetime(d_val)
    d_key = int(dt.strftime("%Y%m%d"))
    if d_key not in existing_date_keys:
        date_rows.append((
            d_key, dt.date(), dt.year, dt.quarter, dt.month,
            dt.strftime("%B"), dt.isoweekday(), dt.strftime("%A"),
            1 if dt.isoweekday() in [6, 7] else 0
        ))
        existing_date_keys.add(d_key)

if date_rows:
    cur.executemany("""
        INSERT INTO dim_date (date_key, full_date, year, quarter, month, month_name, day_of_week, day_name, is_weekend)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, date_rows)
    conn.commit()
print(f"[+] dim_date: Inserted {len(date_rows)} new dates.")

# --- 5.2 Facility Type Dimension ---
cur.execute("SELECT raw_name, facility_type_key FROM dim_facility_type")
fac_lookup = {r[0]: r[1] for r in cur.fetchall()}

fac_rows = []
for raw_fac in df_inspections['facility_type'].fillna('UNKNOWN').unique():
    raw_fac_clean = clean_text(raw_fac)[:150]
    if raw_fac_clean not in fac_lookup:
        fac_rows.append((raw_fac_clean, standardize_facility_type(raw_fac_clean)))

if fac_rows:
    cur.executemany("INSERT INTO dim_facility_type (raw_name, standardized_category) VALUES (?, ?)", fac_rows)
    conn.commit()
    cur.execute("SELECT raw_name, facility_type_key FROM dim_facility_type")
    fac_lookup = {r[0]: r[1] for r in cur.fetchall()}
print(f"[+] dim_facility_type: Loaded {len(fac_lookup)} total facility types.")

# --- 5.3 Geography Dimension ---
cur.execute("SELECT street_address, COALESCE(zip_code, ''), geography_key FROM dim_geography")
geo_lookup = {(r[0], r[1]): r[2] for r in cur.fetchall()}

geo_rows = []
for _, row in df_inspections.iterrows():
    addr = clean_text(row['address']) or "UNKNOWN ADDRESS"
    zip_val = str(row['zip_code'])[:5] if pd.notna(row['zip_code']) else ""
    key = (addr, zip_val)
    if key not in geo_lookup:
        lat, lon = validate_coordinates(row['latitude'], row['longitude'])
        geo_rows.append((addr, "CHICAGO", "IL", zip_val or None, lat, lon))
        geo_lookup[key] = -1

if geo_rows:
    cur.executemany("""
        INSERT INTO dim_geography (street_address, city, state, zip_code, latitude, longitude)
        VALUES (?, ?, ?, ?, ?, ?)
    """, geo_rows)
    conn.commit()
    cur.execute("SELECT street_address, COALESCE(zip_code, ''), geography_key FROM dim_geography")
    geo_lookup = {(r[0], r[1]): r[2] for r in cur.fetchall()}
print(f"[+] dim_geography: Loaded {len(geo_lookup)} total geographic locations.")

# --- 5.4 Establishment Dimension ---
cur.execute("SELECT license_number, dba_name, establishment_key FROM dim_establishment")
estab_lookup = {(r[0], r[1]): r[2] for r in cur.fetchall()}

estab_rows = []
for _, row in df_inspections.iterrows():
    lic = int(row['license_number']) if pd.notna(row['license_number']) else 0
    dba = clean_text(row['dba_name']) or "UNKNOWN"
    aka = clean_text(row['aka_name']) or dba
    risk = standardize_risk(row['risk_level'])
    d_str = str(row['inspection_date'])[:10] if pd.notna(row['inspection_date']) else "2000-01-01"
    
    key = (lic, dba)
    if key not in estab_lookup:
        estab_rows.append((lic, dba, aka, risk, d_str, d_str))
        estab_lookup[key] = -1

if estab_rows:
    cur.executemany("""
        INSERT INTO dim_establishment (license_number, dba_name, aka_name, risk_level, first_inspection_date, last_inspection_date)
        VALUES (?, ?, ?, ?, ?, ?)
    """, estab_rows)
    conn.commit()
    cur.execute("SELECT license_number, dba_name, establishment_key FROM dim_establishment")
    estab_lookup = {(r[0], r[1]): r[2] for r in cur.fetchall()}
print(f"[+] dim_establishment: Loaded {len(estab_lookup)} total unique establishments.")

conn.close()


[+] dim_date: Inserted 0 new dates.
[+] dim_facility_type: Loaded 86 total facility types.
[+] dim_geography: Loaded 5681 total geographic locations.
[+] dim_establishment: Loaded 6863 total unique establishments.


### 6. Populate Fact Tables (`fact_inspections` and `fact_violations`)
link each inspection to its foreign keys, aggregate violation counts by severity, and load them into SQL Server.


In [7]:
conn = get_connection()
cur = conn.cursor()
cur.fast_executemany = True

# Pre-aggregate violation counts per inspection from df_violations
viol_counts = {}
for _, v in df_violations.iterrows():
    insp_id = int(v['inspection_id'])
    tier = str(v['severity_tier'])
    if insp_id not in viol_counts:
        viol_counts[insp_id] = {'priority': 0, 'serious': 0, 'core': 0, 'total': 0}
    viol_counts[insp_id]['total'] += 1
    if 'Priority (Critical)' in tier:
        viol_counts[insp_id]['priority'] += 1
    elif 'Priority Foundation' in tier:
        viol_counts[insp_id]['serious'] += 1
    else:
        viol_counts[insp_id]['core'] += 1

cur.execute("SELECT inspection_id FROM fact_inspections")
existing_fact_ids = {r[0] for r in cur.fetchall()}

fact_rows = []
for _, row in df_inspections.iterrows():
    insp_id = int(row['inspection_id'])
    if insp_id in existing_fact_ids:
        continue

    dt = pd.to_datetime(row['inspection_date'])
    date_key = int(dt.strftime("%Y%m%d")) if pd.notna(row['inspection_date']) else 20000101

    lic = int(row['license_number']) if pd.notna(row['license_number']) else 0
    dba = clean_text(row['dba_name']) or "UNKNOWN"
    estab_key = estab_lookup.get((lic, dba))

    addr = clean_text(row['address']) or "UNKNOWN ADDRESS"
    zip_val = str(row['zip_code'])[:5] if pd.notna(row['zip_code']) else ""
    geo_key = geo_lookup.get((addr, zip_val))

    raw_fac = clean_text(row['facility_type']) or "UNKNOWN"
    fac_key = fac_lookup.get(raw_fac)

    insp_type = standardize_inspection_type(row['inspection_type'])
    result_val = str(row['results']).strip().title() if pd.notna(row['results']) else "Unknown"
    is_fail = 1 if result_val == "Fail" else 0

    vc = viol_counts.get(insp_id, {'priority': 0, 'serious': 0, 'core': 0, 'total': 0})

    fact_rows.append((
        insp_id, date_key, estab_key, geo_key, fac_key,
        insp_type, result_val, is_fail,
        vc['priority'], vc['serious'], vc['core'], vc['total'],
        None, 0
    ))

if fact_rows:
    cur.executemany("""
        INSERT INTO fact_inspections (
            inspection_id, date_key, establishment_key, geography_key, facility_type_key,
            inspection_type, results, is_failure,
            priority_violations_count, serious_violations_count, core_violations_count, total_violations_count,
            days_since_last_inspection, prior_failures_cumulative
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, fact_rows)
    conn.commit()
    print(f"[+] fact_inspections: Inserted {len(fact_rows)} inspection fact records.")
else:
    print("[+] fact_inspections: All records already up to date.")

# --- Populate fact_violations ---
cur.execute("SELECT DISTINCT inspection_id FROM fact_violations")
existing_v_ids = {r[0] for r in cur.fetchall()}

# Only insert violations for valid inspections
cur.execute("SELECT inspection_id FROM fact_inspections")
valid_inspections = {r[0] for r in cur.fetchall()}

viol_fact_rows = []
for _, v in df_violations.iterrows():
    insp_id = int(v['inspection_id'])
    if insp_id in valid_inspections and insp_id not in existing_v_ids:
        code_num = int(v['violation_code']) if pd.notna(v['violation_code']) else 0
        desc = str(v['violation_description'])[:255] if pd.notna(v['violation_description']) else ""
        tier = str(v['severity_tier']) if pd.notna(v['severity_tier']) else "Core (Minor)"
        weight = 3 if 'Critical' in tier else (2 if 'Serious' in tier else 1)
        comment = str(v['inspector_comment']) if pd.notna(v['inspector_comment']) else ""
        
        viol_fact_rows.append((insp_id, code_num, desc, tier, weight, comment))

if viol_fact_rows:
    # Batch insert chunks of 5000
    chunk_size = 5000
    for i in range(0, len(viol_fact_rows), chunk_size):
        chunk = viol_fact_rows[i:i + chunk_size]
        cur.executemany("""
            INSERT INTO fact_violations (
                inspection_id, violation_code, violation_description, severity_tier, severity_weight, inspector_comment
            )
            VALUES (?, ?, ?, ?, ?, ?)
        """, chunk)
    conn.commit()
    print(f"[+] fact_violations: Inserted {len(viol_fact_rows)} granular violation rows.")
else:
    print("[+] fact_violations: All violations already up to date.")

conn.close()


[+] fact_inspections: All records already up to date.
[+] fact_violations: All violations already up to date.


### 7. Verify the Star Schema in SQL Server
Let's query the tables to verify the final dimensional model.


In [9]:
conn = get_connection()

tables = ['dim_date', 'dim_facility_type', 'dim_geography', 'dim_establishment', 'fact_inspections', 'fact_violations']
counts = []

for tbl in tables:
    cnt = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {tbl}", conn).iloc[0]['row_count']
    counts.append({'Table Name': tbl, 'Total Rows': cnt})

df_counts = pd.DataFrame(counts)
print("=== STAR SCHEMA ROW COUNTS IN MICROSOFT SQL SERVER ===")
print(df_counts.to_string(index=False))

# Preview joined Star Schema
df_preview = pd.read_sql("""
SELECT TOP 5
    i.inspection_id,
    d.full_date AS inspection_date,
    e.dba_name AS establishment_name,
    f.standardized_category AS facility_type,
    g.street_address,
    g.zip_code,
    i.inspection_type,
    i.results,
    i.priority_violations_count,
    i.total_violations_count
FROM fact_inspections i
JOIN dim_date d ON i.date_key = d.date_key
JOIN dim_establishment e ON i.establishment_key = e.establishment_key
JOIN dim_facility_type f ON i.facility_type_key = f.facility_type_key
JOIN dim_geography g ON i.geography_key = g.geography_key
ORDER BY d.full_date DESC;
""", conn)

conn.close()

df_preview.to_string(index=False)


=== STAR SCHEMA ROW COUNTS IN MICROSOFT SQL SERVER ===
       Table Name  Total Rows
         dim_date         157
dim_facility_type          86
    dim_geography        5681
dim_establishment        6863
 fact_inspections       10000
  fact_violations       23028


C:\Users\muiva\AppData\Local\Temp\ipykernel_24332\2644035880.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cnt = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {tbl}", conn).iloc[0]['row_count']
C:\Users\muiva\AppData\Local\Temp\ipykernel_24332\2644035880.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_preview = pd.read_sql("""


" inspection_id inspection_date          establishment_name                facility_type    street_address zip_code           inspection_type   results  priority_violations_count  total_violations_count\n       2643249      2026-09-24 JUNIOR'S MARISCOS AND GRILL                   Restaurant 5960 S ARCHER AVE    60638 License Initial / Renewal      Pass                          0                       3\n       2643288      2026-09-24    SIP & SAVOR COFFEE HOUSE Other / General Food Service   230 W MONROE ST    60606 License Initial / Renewal Not Ready                          0                       0\n       2643312      2026-09-24             CROSSOVER GRILL                   Restaurant     835 E 75TH ST    60619 License Initial / Renewal      Pass                          0                       0\n       2643318      2026-09-24             CROSSOVER GRILL                   Restaurant     835 E 75TH ST    60619 License Initial / Renewal      Pass                          0          